# ES4304 — Data Access Accounts

To take part in the course you need accounts with three data providers, to download the satellite data the tutorials use.

**You MUST do this before the course starts.** You cannot process any data without these accounts, and sorting them out during a session wastes your time and everyone else's.

| Provider | Needed for | Register at | Approval |
|---|---|---|---|
| **NASA Earthdata** | PACE (2.1), SWOT (2.3), OSCAR (2.4) | <https://urs.earthdata.nasa.gov/users/new> | Immediate |
| **JAXA P-Tree** | Himawari SST (2.2) | <https://www.eorc.jaxa.jp/ptree/registration_top.html> | **Up to several working days** |
| **Copernicus Data Space** | Sentinel-2 (1.1, 1.2), Sentinel-1 (3.1) | <https://dataspace.copernicus.eu/> | Immediate |

All three are free and quick to fill in. The JAXA one cannot be rushed, though — a person approves it, so register now rather than the night before Tutorial 2.

The more fiddly part is telling the notebooks about your accounts, which is what the rest of this notebook does. **Read each section and edit the code cells carefully.**

## 1. Setup

The same cell appears at the top of every notebook in this course. It installs only what is missing, so it costs nothing when there is nothing to do.

In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# In a Codespace, or a local conda env, everything is already installed and
# this cell does nothing. It installs anything missing, as a safety net.
import importlib.util
import os
import subprocess
import sys

REQUIRED = {                      # import name -> pip name
    "earthaccess": "earthaccess>=0.16",
    "requests": "requests>=2.31",
}

missing = [pip for mod, pip in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=True)
else:
    print("Environment ready - nothing to install.")

## 2. The `.netrc` file

All three providers are reached over the network with a username and password. Rather than typing them every time, you save them once in a file called **`.netrc`** in your home directory. The `earthaccess` library and Python's `netrc` module both know to look there, so from then on the tutorial notebooks log in without asking you anything.

`.netrc` is a plain text file, one line per machine:

```
machine <hostname> login <username> password <password>
```

It lives in your **home directory**, which is outside this repository:

| Where you are running | The file |
|---|---|
| Codespace, Linux, macOS | `~/.netrc` — that is `/home/you/.netrc` or `/Users/you/.netrc` |
| Windows | `C:\Users\you\.netrc` |

The cell below works out the right path for you, so there is nothing to type. In a Codespace the file survives kernel restarts and stopping the codespace — if you delete the codespace and create a new one, run this notebook again.

> **The one thing to be careful about.** You are about to type your passwords into code cells. The `.netrc` file itself is fine, but a **saved notebook keeps whatever you typed into a cell** — and this notebook is in a Git repository. Section 4 puts the placeholders back for you when you are done; do not skip it.

### The cell that writes it

Run this cell once, before the three below. It defines `save_credentials()`, which the Earthdata, P-Tree and Copernicus cells then call — one line of `.netrc` each.

It is plain Python rather than a shell command on purpose. `echo ... > $HOME/.netrc` is a Unix shell line: on Windows there is no `$HOME` and no `chmod`, so that version writes a file called `$HOME` in the wrong place, if it works at all. `Path.home()` is right on every platform.

In [ ]:
# --- Run this first. It writes nothing on its own; the three cells below do. --
import netrc
import os
import stat
from pathlib import Path

# Path.home() is the one line that makes this work everywhere: /home/you on
# Linux and in a codespace, /Users/you on macOS, C:\Users\you on Windows.
NETRC = Path.home() / ".netrc"


def save_credentials(machine, login, password):
    """Add or replace one machine's line in ~/.netrc, leaving the others alone.

    Re-running a cell is safe. The old line for that machine is replaced rather
    than duplicated, and the other providers' lines stay where they are - so
    the order you run the three cells below in does not matter.

    Written as UTF-8, which is what Python's netrc module tries first when it
    reads the file back, on Windows as much as anywhere else.
    """
    keep = []
    if NETRC.exists():
        for line in NETRC.read_text(encoding="utf-8").splitlines():
            fields = line.split()
            if not fields or fields[:2] == ["machine", machine]:
                continue                  # drop blanks and this machine's line
            keep.append(line)

    previously = NETRC.read_text(encoding="utf-8") if NETRC.exists() else None
    keep.append(f"machine {machine} login {login} password {password}")
    NETRC.write_text("\n".join(keep) + "\n", encoding="utf-8")

    # 600 - readable by you and nobody else. Some programs refuse to read a
    # .netrc that anyone else can. Windows has no such permission bits, and
    # Python's netrc module only checks them on Linux and macOS, so there is
    # nothing to set there.
    if os.name != "nt":
        NETRC.chmod(stat.S_IRUSR | stat.S_IWUSR)

    # Read it straight back, because `.netrc` cannot hold every password.
    # One with a space in it does not parse at all, and a backslash is quietly
    # dropped - which would show up much later as a wrong password, in the
    # middle of a tutorial, with nothing to point at. If it did not survive the
    # round trip, put the file back as it was and say so now.
    try:
        stored = netrc.netrc(NETRC).authenticators(machine)
    except netrc.NetrcParseError:
        stored = None

    if stored is None or stored[0] != login or stored[2] != password:
        if previously is None:
            NETRC.unlink()
        else:
            NETRC.write_text(previously, encoding="utf-8")
        raise ValueError(
            f".netrc cannot hold this password, so nothing was saved for "
            f"{machine}.\nA space in a password stops the file parsing, a "
            f"backslash is dropped silently,\nand an accented character works "
            f"only on newer Pythons. Change the password\nat the provider to "
            f"one made of ordinary characters, then run this cell again.\n"
            f"The rest of your .netrc is untouched.")

    print(f"Saved {machine} to {NETRC}")

### NASA Earthdata

Getting data from any NASA Distributed Active Archive Center (DAAC) requires a free Earthdata account. Register — or log in, if you already have one — at the Earthdata User Registration Service: <https://urs.earthdata.nasa.gov/home>. You are sent to this same site whenever you download NASA data through a browser.

Substitute **your** Earthdata username for `myUsername` and **your** password for `myPassword` below, then run the cell.

In [ ]:
# if you don't substitute your own details here, this won't work
save_credentials("urs.earthdata.nasa.gov", "myUsername", "myPassword")

### JAXA P-Tree

P-Tree is JAXA's Himawari data portal. When your registration is approved you receive an **FTP username and password** — these are *separate from your P-Tree website login*, which is the single most common thing to get wrong here. The FTP username is usually your email address with the `@` replaced by an underscore.

Details and FAQ: <https://www.eorc.jaxa.jp/ptree/faq.html>

Substitute your P-Tree **FTP** credentials below and run it. This adds a second line and leaves the Earthdata line alone.

**If your approval has not arrived yet, skip this cell** and come back to it. Nothing before Tutorial 2.2 needs it.

In [ ]:
# again, if you don't edit this to use your own details, this will not work
save_credentials("ftp.ptree.jaxa.jp", "myEmail_example.com", "myPtreePassword")

### Copernicus Data Space Ecosystem

The Copernicus Data Space Ecosystem (CDSE) is ESA's portal for Sentinel data, which the Tutorial 1 notebooks download as `.SAFE` products. Register at <https://dataspace.copernicus.eu/>: click the avatar at the top right, then **REGISTER**, and confirm the email it sends you. There is no waiting for a human — you can use it straight away.

Your login here is the **email address you registered with**, and the password is the one you chose on the form. Unlike P-Tree, there is no separate download account.

Substitute your own below and run it. This adds a third line and leaves the other two alone.

> **If you have turned on two-factor authentication** for this account, the check in section 3 needs the six-digit code from your authenticator app as well; there is a commented-out line in that cell for it. Nothing else changes.

In [ ]:
# and again here - the email address you registered with, and its password
save_credentials("identity.dataspace.copernicus.eu", "myEmail@example.com", "myCopernicusPassword")

### About the file permissions

On Linux and macOS `save_credentials()` also sets the file's permissions to `600` — readable by you and nobody else. Some programs check this and refuse to read a `.netrc` that anyone else can. Windows has no equivalent bits and Python does not check for them there, so nothing happens on Windows and nothing needs to.

## 3. Check it worked

First, look at what is in the file. This prints the hostnames and usernames, and masks the passwords — so that if you save the notebook with this output in it, nothing leaks.

In [ ]:
import netrc

nrc = netrc.netrc()

for machine in sorted(nrc.hosts):
    login, _, password = nrc.hosts[machine]
    print(f"{machine:34s} login={login!r:30s} password={'*' * len(password)}")

### NASA Earthdata

`earthaccess.login(strategy="netrc")` reads the file and logs in against Earthdata. **This route checks your credentials immediately** — a wrong password raises here, rather than failing later in the middle of a tutorial.

Then we download one real file, about 26 MB, and delete it again. Logging in proves the password is right; downloading proves the account can actually get data.

In [ ]:
import shutil

import earthaccess

auth = earthaccess.login(strategy="netrc")

# `authenticated` is the flag that actually means "logged in". `username` is
# only filled in when the login used a username and password - a token login
# leaves it as None, so printing it alone is misleading.
print("Authenticated:", auth.authenticated)
print("Logged in to Earthdata as:", auth.username or "(token login - no username)")

if auth.username is None:
    print("\nNo username means earthaccess did not use your ~/.netrc entry - it")
    print("was already authenticated with an EARTHDATA_TOKEN. Downloads will")
    print("work, but the credentials you saved above have NOT been checked.")
    print("Print it with os.environ.get('EARTHDATA_TOKEN'); if that shows")
    print("anything, remove it from your Codespaces secrets, then restart the")
    print("kernel and re-run this notebook to test the .netrc credentials.")

results = earthaccess.search_data(
    short_name="PACE_OCI_L3M_BGC",     # chlorophyll lives in the biogeochemistry suite
    version="3.2",
    granule_name="*MO*0p1*",           # monthly, 0.1 degree
    count=1,
)

if not results:
    print("\nSearch found nothing. That is not an account problem - a retired")
    print("short_name returns an empty list rather than an error. See S01.")
else:
    print("Found:", results[0].data_links()[0].split("/")[-1])

    TEST_DIR = os.path.join(os.getcwd(), "netrc_check")
    try:
        files = earthaccess.download(results, TEST_DIR)
        print("Downloaded:", os.path.basename(files[0]))
        print("\nNASA Earthdata access works.")
    finally:
        shutil.rmtree(TEST_DIR, ignore_errors=True)

### JAXA P-Tree

`ftplib` does not read `.netrc` by itself, so we use the standard library's `netrc` module to pull the credentials out and hand them over. Every notebook that needs P-Tree does exactly this, which is why you never type the password again.

Skip this cell if your approval has not arrived.

In [ ]:
import netrc
from ftplib import FTP

FTP_SERVER = "ftp.ptree.jaxa.jp"

ftp_user, _, ftp_password = netrc.netrc().authenticators(FTP_SERVER)

with FTP(FTP_SERVER) as ftp:
    ftp.login(ftp_user, ftp_password)
    print("Connected. Top-level directories:", ftp.nlst()[:5])
    print("\nJAXA P-Tree access works.")

### Copernicus Data Space Ecosystem

CDSE does not take your password on every request. You exchange it for a short-lived **access token**, and downloads carry the token instead — which is why the notebooks that use Sentinel data all start with the same few lines below, reading your credentials out of `.netrc` exactly as the P-Tree cell does.

The exchange is where a wrong password shows up, as `401 Unauthorized`. Searching the catalogue needs no account at all, so — as with Earthdata — we then read the first bytes of a real product, which does.

In [ ]:
import netrc

import requests

CDSE_HOST = "identity.dataspace.copernicus.eu"
TOKEN_URL = f"https://{CDSE_HOST}/auth/realms/CDSE/protocol/openid-connect/token"

cdse_user, _, cdse_password = netrc.netrc().authenticators(CDSE_HOST)

response = requests.post(TOKEN_URL, timeout=60, data={
    "client_id": "cdse-public",       # the public client every CDSE script uses
    "grant_type": "password",
    "username": cdse_user,
    "password": cdse_password,
    # With two-factor authentication on, uncomment this and put in the current
    # six-digit code from your authenticator app before running the cell.
    # "totp": "123456",
})
response.raise_for_status()           # 401 here means the password is wrong
token = response.json()["access_token"]

print("Logged in to Copernicus Data Space as:", cdse_user)

# The catalogue is open to anyone; downloading is the part that needs the
# account. Take the newest Sentinel-2 scene there is ...
CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

product = requests.get(CATALOGUE, timeout=60, params={
    "$filter": "Collection/Name eq 'SENTINEL-2' and contains(Name,'MSIL2A')",
    "$orderby": "ContentDate/Start desc",
    "$top": 1,
}).json()["value"][0]

print("Found:", product["Name"])

# ... and read the first kilobyte of it. A Sentinel-2 scene is about a
# gigabyte; `stream=True` means nothing is fetched until we ask, and leaving
# the `with` block closes the connection, so the rest never arrives.
#
# Note the hostname. `download` is the download service - `zipper` is an older
# name for the same thing, and you will see it in some notebooks. Do not ask
# the `catalogue` host for the file: it redirects here, and `requests` drops
# the Authorization header when a redirect crosses hostnames, so the token
# would be thrown away and you would get a 401 that is nothing to do with you.
DOWNLOAD = "https://download.dataspace.copernicus.eu/odata/v1/Products({0})/$value"

with requests.get(DOWNLOAD.format(product["Id"]), timeout=60, stream=True,
                  headers={"Authorization": f"Bearer {token}"}) as file_stream:
    file_stream.raise_for_status()
    head = next(file_stream.iter_content(1024))

print(f"Read the first {len(head)} bytes of it.")
print("\nCopernicus Data Space access works.")

## 4. Tidy up before you commit

Your `~/.netrc` is in your home directory, not in this repository, so it is never committed and keeps working after this.

**The code cells above are in the repository.** Run the cell below: it rewrites this notebook file with the placeholder text back in place and clears the outputs, so your passwords are not saved into it.

In [ ]:
# Restores the placeholders in this notebook's own file. Your ~/.netrc is
# untouched and keeps working.
import json
import re

NB_PATH = "0.2.1_Account_Check.ipynb"

# What the three write cells hold before anyone types anything into them.
PLACEHOLDERS = {
    "urs.earthdata.nasa.gov": ("myUsername", "myPassword"),
    "ftp.ptree.jaxa.jp": ("myEmail_example.com", "myPtreePassword"),
    "identity.dataspace.copernicus.eu": ("myEmail@example.com",
                                         "myCopernicusPassword"),
}

BLANK = {machine: f'save_credentials("{machine}", "{login}", "{password}")'
         for machine, (login, password) in PLACEHOLDERS.items()}

# A line that *is* a call - not the `def` in the helper cell, and not the
# string just above.
CALL = re.compile(r"^[ \t]*save_credentials\(", re.MULTILINE)


def end_of_call(text, start):
    """Index just past the `)` that closes the call at `start`, or None.

    Scanned for rather than matched with a pattern, so the call is replaced
    whole whatever is inside it - a password with a bracket, a quote, a comma
    or a line break in it included.
    """
    depth = 0
    quote = ""
    for i in range(text.index("(", start), len(text)):
        character = text[i]
        if quote:                            # inside a string: nothing counts
            quote = "" if character == quote else quote
        elif character in "\"'":
            quote = character
        elif character == "(":
            depth += 1
        elif character == ")":
            depth -= 1
            if depth == 0:
                return i + 1
    return None


def end_of_line(text, index):
    """Index of the newline ending the line `index` is on, or of the end."""
    end = text.find("\n", index)
    return len(text) if end == -1 else end


def restore(text):
    """`text` with every save_credentials() call put back to its placeholder."""
    pieces, cursor, unknown = [], 0, []
    while True:
        found = CALL.search(text, cursor)
        if found is None:
            break

        start = found.end() - len("save_credentials(")
        # A call with no closing bracket at all - half typed, or one deleted -
        # is taken to end with its line, so that its password still goes.
        end = end_of_call(text, start) or end_of_line(text, start)
        machine = next((m for m in PLACEHOLDERS if m in text[start:end]), None)

        pieces.append(text[cursor:start])
        if machine is None:
            pieces.append(text[start:end])   # some other host: we do not know
            unknown.append(" ".join(text[start:end].split())[:70])
        else:
            pieces.append(BLANK[machine])
            # An unescaped quote in a password ends its string early, so the
            # bracket found above can be one from inside the password, leaving
            # the rest of it on the line. Anything left there that is not a
            # comment is wreckage of that kind, and goes with the call.
            tail = end_of_line(text, end)
            trailing = text[end:tail]
            if trailing.strip() and not trailing.lstrip().startswith("#"):
                end = tail
        cursor = end

    pieces.append(text[cursor:])
    return "".join(pieces), unknown


with open(NB_PATH, encoding="utf-8") as fh:
    nb = json.load(fh)

changed, unknown, broken = 0, [], []
for cell in nb["cells"]:
    text = "".join(cell["source"])
    if "save_credentials(" in text and any(m in text for m in PLACEHOLDERS):
        text, odd = restore(text)
        unknown.extend(odd)
        if text != "".join(cell["source"]):
            cell["source"] = text.splitlines(keepends=True)
            changed += 1
        try:
            # A cell that no longer parses was edited into some shape none of
            # the above understands, and may still be holding a password.
            compile(text, "<cell>", "exec")
        except SyntaxError as error:
            broken.append((error.msg, (error.text or "").strip()))
    if cell["cell_type"] == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

with open(NB_PATH, "w", encoding="utf-8") as fh:
    json.dump(nb, fh, indent=1, ensure_ascii=False)
    fh.write("\n")

if changed:
    print(f"Placeholders restored in {changed} cell(s); outputs cleared.")
    print("Close and reopen this notebook so the editor picks up the change.")
else:
    print("Nothing needed replacing - the placeholders were already back.")
    print("Outputs cleared; close and reopen the notebook to see that.")

for call in unknown:
    print("\nLeft alone - not a provider this notebook knows about:")
    print("   ", call)

for message, line in broken:
    print("\nCHECK THIS BY HAND BEFORE YOU COMMIT. A write cell does not read")
    print(f"as Python any more ({message}), so it may still hold a password:")
    print("   ", line)

## If something failed

- **`FileNotFoundError`, or "No .netrc found".** The write cells did not run, or you are in a brand-new codespace. Run section 2 again.
- **`NameError: name 'save_credentials' is not defined`.** Run the helper cell at the top of section 2 first — the three write cells call it.
- **`LoginAttemptFailure`.** Earthdata rejected the username or password. Check them by logging in at <https://urs.earthdata.nasa.gov>, then re-run the Earthdata write cell. It replaces only the Earthdata line, so your P-Tree and Copernicus lines are left alone.
- **"Logged in to Earthdata as: None", or "(token login - no username)".** You are authenticated, but not with the `.netrc` entry you just wrote — an `EARTHDATA_TOKEN` in the environment got there first, and `earthaccess` skips the `.netrc` lookup once it is already logged in. Downloads work, but this notebook has not verified your username and password. Print the variable — `import os; print(os.environ.get("EARTHDATA_TOKEN"))` — and if it shows anything, remove it from your Codespaces secrets (**Settings → Codespaces → Repository secrets**), then rebuild or restart the kernel and re-run.
- **`401 Unauthorized`** from the Copernicus token cell, or `"Token not found"`. The email address or password is wrong — check them by logging in at <https://dataspace.copernicus.eu/> — or the account has two-factor authentication on, in which case the request needs the `totp` line in that cell as well.
- **`530 Login incorrect`** from JAXA. You used your website login rather than the FTP credentials, or your registration is not approved yet.
- **`TypeError: cannot unpack non-sequence NoneType`** in a check cell. `.netrc` has no line for that host — `ftp.ptree.jaxa.jp` or `identity.dataspace.copernicus.eu` — so run the write cell for it.
- **`ValueError: .netrc cannot hold this password`.** Exactly what it says: the file format cannot store a password with a space in it, quietly loses a backslash, and only handles accented characters on newer Pythons. Nothing was saved, and the rest of the file is as it was — change the password at the provider and run the cell again.
- **`SyntaxError`** in a write cell. Your password contains a double quote, which ends the Python string early. Wrap that argument in single quotes instead: `'my"password'`.
- Anything else: [S01 Troubleshooting](../S01_Troubleshooting/README.md).

## Next

[Tutorial 2 overview](../2.0_Tutorial_2_Overview_and_Assignment/README.md)